# 01 — Environment Setup

Verify dependencies, CUDA GPU, and HuggingFace access.

**Colab:** Enable GPU (Runtime → Change runtime type → T4 GPU) and add `HF_TOKEN` in Secrets (🔑).

In [ ]:
# Run once on Colab
import os
if os.path.exists('/content'):
    get_ipython().system('pip install -q -r /content/ABPSEES/requirements.txt')
    get_ipython().system('pip install -q -e /content/ABPSEES')


In [ ]:
import os
import sys
from pathlib import Path

# Colab: project lives at /content/ABPSEES
if Path('/content/ABPSEES/project').exists():
    ROOT = Path('/content/ABPSEES')
else:
    ROOT = Path.cwd()
    if not (ROOT / 'project').exists():
        ROOT = ROOT.parent

sys.path.insert(0, str(ROOT))
os.environ['ABPSEES_ROOT'] = str(ROOT)

%load_ext autoreload
%autoreload 2

from project.config import load_config, is_colab
from project.utils import setup_logging, setup_colab_environment

setup_logging()
config = load_config()  # auto-selects colab.yaml on Colab
setup_colab_environment(
    hf_token_env=config.colab.hf_token_env,
    use_colab_secrets=config.colab.use_colab_secrets,
    mount_google_drive=config.colab.mount_google_drive,
)

print('Project root:', ROOT)
print('Colab:', is_colab())
print('Data dir:', config.paths.data_dir)


In [ ]:
import importlib
import torch

packages = [
    'torch', 'transformers', 'peft', 'accelerate', 'datasets', 'bitsandbytes',
    'sklearn', 'pandas', 'numpy', 'matplotlib', 'yaml', 'scipy',
]
for pkg in packages:
    try:
        importlib.import_module(pkg)
        print(f'OK  {pkg}')
    except ImportError as exc:
        print(f'FAIL {pkg}: {exc}')

print('CUDA available:', torch.cuda.is_available())
if torch.cuda.is_available():
    print('GPU:', torch.cuda.get_device_name(0))
    print('VRAM (GB):', round(torch.cuda.get_device_properties(0).total_memory / 1e9, 1))


In [ ]:
from project.utils import get_device_config, set_seed

device_cfg = get_device_config(require_cuda=is_colab())
set_seed(config.seed)
print('Device:', device_cfg.device)
print('QLoRA:', device_cfg.use_qlora)


In [ ]:
from project.model import load_tokenizer

tokenizer = load_tokenizer(config)
print('Tokenizer:', tokenizer.__class__.__name__)
